# Cantilever beam — crack diagnosis end-to-end

Same three-head SHM workflow as the LANL benchmark, repointed at a single
cantilever beam with a simulated crack. Same `pymodal` library plumbing —
`Scenario`, `ParameterVariation`, `build_frf_collection` — and an analogous
reduced-order modal model in `cantilever_model.py`.

| step | what it does |
|---|---|
| 1 | build the beam: a 0.5 m steel cantilever, 25 mm × 5 mm cross-section, 60 Bernoulli-Euler beam elements, fixed at x = 0, free at x = L |
| 2 | enumerate 25 damage scenarios = 1 pristine + 6 crack positions × 4 crack depths |
| 3 | shaker at x = 0.05 L (transverse Z); ten transverse accelerometers at x = 0.1 L … 1.0 L |
| 4 | build a 2 500-sample labelled `pymodal.frf` collection (100 perturbed realisations × 25 classes), persisted to a single HDF5 file |
| 5 | train **three** heads on the same FRFs:<br>• detection — *classifier* (pristine vs cracked)<br>• localisation — *regression* on the distance from the fixed end<br>• severity — *regression* on the crack depth |
| 6 | report a binary confusion matrix plus two predicted-vs-true scatters |

Crack model: the FEM element containing the crack has its flexural
stiffness reduced by `(1 - d/h)³` (broken-section approximation). This is
the same surrogate physics that the LANL reduced-order model uses for
column thinning, so both demos share the same library glue.


## Setup


In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle, FancyArrowPatch

HERE = Path(os.getcwd()).resolve()
EXAMPLE_DIR = HERE / "examples" / "cantilever_crack" if (HERE / "examples").exists() else HERE
REPO_ROOT = EXAMPLE_DIR.parent.parent
for p in (str(REPO_ROOT), str(EXAMPLE_DIR)):
    if p not in sys.path:
        sys.path.insert(0, p)

import params as P
import cantilever_model as CM
import damage_scenarios as DS
import pymodal

print("pymodal", pymodal.__version__,
      "  beam:", f"{P.BEAM_L*1000:.0f} x {P.BEAM_B*1000:.0f} x {P.BEAM_H*1000:.0f} mm",
      "  scenarios =", 1 + len(P.DAMAGE_POSITIONS) * len(P.DAMAGE_DEPTHS))


## 1. Geometry, sensors, shaker

Side view of the beam. The crack is shown as a vertical notch in the upper
face at ``x_c``, with a depth proportional to ``d/h``. Pristine samples
have no notch.


In [ ]:
def plot_beam(geom, ax=None, title=None, show_sensors=True):
    if ax is None:
        fig, ax = plt.subplots(figsize=(11, 2.2))
    L, h = geom.L, geom.h
    # beam body (thicker in the plot than reality, for visibility)
    visual_h = 0.05 * L
    ax.add_patch(Rectangle((0, -visual_h/2), L, visual_h,
                             facecolor='lightsteelblue', edgecolor='steelblue', linewidth=1.5))
    # fixed-end hatch
    ax.add_patch(Rectangle((-0.02 * L, -visual_h*0.9), 0.02 * L, visual_h*1.8,
                             facecolor='dimgrey', edgecolor='black', hatch='///'))
    # crack
    if geom.crack_depth_frac > 0:
        x_c = geom.crack_position_frac * L
        crack_depth_visual = visual_h * geom.crack_depth_frac
        ax.add_patch(Rectangle((x_c - 0.002 * L, visual_h/2 - crack_depth_visual),
                                 0.004 * L, crack_depth_visual,
                                 facecolor='black', edgecolor='black', zorder=5))
    # shaker arrow
    x_shaker = P.SHAKER_FRAC * L
    ax.annotate('', xy=(x_shaker, -visual_h/2 - 0.005*L),
                xytext=(x_shaker, -visual_h/2 - 0.05*L),
                arrowprops=dict(arrowstyle='->', color='magenta', lw=2))
    ax.text(x_shaker, -visual_h/2 - 0.06*L, 'shaker',
             ha='center', va='top', color='magenta', fontsize=9)
    # sensors (green dots above the beam top edge)
    if show_sensors:
        for f in P.SENSOR_FRACS:
            x = f * L
            ax.scatter([x], [visual_h/2 + 0.01*L], color='limegreen', s=40,
                        edgecolors='black', linewidths=0.5, zorder=10)
    if title:
        ax.set_title(title)
    ax.set_xlim(-0.05 * L, 1.05 * L); ax.set_ylim(-0.10 * L, 0.10 * L)
    ax.set_xlabel('x [m]'); ax.set_aspect('equal')
    ax.set_yticks([])

# pristine
geom_pristine = CM.BeamGeometry()
# moderate crack at 40 % of L, depth 40 % of h
geom_cracked = CM.BeamGeometry(crack_position_frac=0.40, crack_depth_frac=0.40)

fig, axes = plt.subplots(2, 1, figsize=(11, 4.4))
plot_beam(geom_pristine, ax=axes[0], title='pristine')
plot_beam(geom_cracked, ax=axes[1],
            title=f'crack at x = {0.40*P.BEAM_L*1000:.0f} mm, depth = {0.40*P.BEAM_H*1000:.2f} mm  (40 % of h)')
legend = [Line2D([0],[0], marker='o', color='w', markerfacecolor='limegreen', markersize=8,
                  label='accelerometers (Z)'),
          Line2D([0],[0], color='magenta', lw=2, label='shaker (Z)'),
          Line2D([0],[0], color='black', lw=4, label='crack')]
fig.legend(handles=legend, loc='lower center', ncol=3, fontsize=9, frameon=False,
            bbox_to_anchor=(0.5, -0.02))
plt.tight_layout(); plt.show()

# pristine modal preview
freqs, _ = CM.modes(geom_pristine)
print('first 4 nat freqs (Hz) pristine :', np.round(freqs[:4], 2))
freqs_d, _ = CM.modes(geom_cracked)
print('first 4 nat freqs (Hz) cracked  :', np.round(freqs_d[:4], 2))


## 2. Damage scenarios

For every crack position ``x_c / L`` we step through four crack depths
``d / h``, plus one pristine class. **25 classes** total.


In [ ]:
scenarios = DS.cantilever_scenarios()
is_dmg, pos_label, depth_label = DS.scenario_meta(scenarios)

print(f"{len(scenarios)} scenarios:")
for sc in scenarios[:5] + scenarios[-5:]:
    print(f"  label {sc.label:>2d}  {sc.name}")
print()
print(f"crack positions x_c/L : {P.DAMAGE_POSITIONS}")
print(f"crack depths    d/h   : {P.DAMAGE_DEPTHS}")


## 3. Inputs / outputs


In [ ]:
inputs  = DS.shaker_input()
outputs = DS.sensor_outputs()
freq_axis = np.arange(P.F_MIN, P.F_MAX + P.F_STEP / 2, P.F_STEP)
print(f"{len(inputs)} input  : {inputs[0]}")
print(f"{len(outputs)} outputs: first {outputs[0]}, last {outputs[-1]}")
print(f"freq axis : {len(freq_axis)} pts, {freq_axis[0]} - {freq_axis[-1]} Hz")


## 4. Build the labelled FRF collection

100 perturbed realisations per scenario × 25 scenarios = **2 500 samples**.
The crack location and depth are kept fixed within each scenario; only the
beam dimensions, modulus, density and damping are jittered with small
Gaussian noise (the realistic interpretation of "manufacturing tolerance"
that does not invalidate the regression labels).


In [ ]:
dataset_path = EXAMPLE_DIR / "cantilever_dataset.h5"
if dataset_path.exists():
    dataset_path.unlink()

frf_collection = DS.build_dataset(
    scenarios       = scenarios,
    n_per_scenario  = 100,
    inputs          = inputs,
    outputs         = outputs,
    freq_array      = freq_axis,
    path            = dataset_path,
    seed            = 0,
    progress        = False,
)
print(f"built {len(frf_collection)} items, "
      f"item shape {frf_collection.measurements[0].shape}, "
      f"file size {dataset_path.stat().st_size / 1e6:.1f} MB")


## 5. Sanity-check class separability

Mean magnitude FRF at the **tip** sensor (x = L) for the pristine class
and for cracks at ``x_c = 0.4 L`` at each of the four severity levels.


In [ ]:
mag = np.stack([np.abs(frf_collection.measurements[i][:, -1, 0])
                  for i in range(len(frf_collection))])
labels = np.array([float(frf_collection.labels[i][()]) for i in range(len(frf_collection))])

# pick the row of scenarios with crack at x/L = 0.40
focus = [0]                    # pristine
target_pos = 0.40
for sc in scenarios:
    if sc.name == 'pristine':
        continue
    parts = sc.name.split('_')
    p_idx = int(parts[0][1:])
    if abs(P.DAMAGE_POSITIONS[p_idx] - target_pos) < 1e-6:
        focus.append(sc.label)

fig, ax = plt.subplots(figsize=(10, 4.5))
cmap = plt.get_cmap('viridis')
for k, lbl in enumerate(focus):
    sel = labels == lbl
    sc_name = scenarios[lbl].name
    ax.semilogy(freq_axis, mag[sel].mean(0),
                 color=cmap(k / max(len(focus) - 1, 1)),
                 label=sc_name, linewidth=1.4)
ax.set_xlabel('frequency [Hz]'); ax.set_ylabel('|H| [mm/s²/N]')
ax.set_title(f'mean |FRF| at tip sensor — varying severity at x_c = {target_pos:.2f} L')
ax.legend(fontsize=8); plt.tight_layout(); plt.show()


## 6. Three diagnostic heads

All three heads share the **same FRF features** (globally normalised
log-magnitude + unwrapped phase per channel) and the **same train/val/test
split** (stratified on the 25-way scenario label).


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

train_idx, val_idx, test_idx = frf_collection.split(0.70, 0.15, 0.15, seed=0)
print(f"split: {len(train_idx)} train / {len(val_idx)} val / {len(test_idx)} test")

n = len(frf_collection)
sample_label = np.array([int(frf_collection.labels[i][()]) for i in range(n)])
sample_is_dmg = is_dmg[sample_label]
sample_pos    = pos_label[sample_label]      # regression target for localisation
sample_depth  = depth_label[sample_label]    # regression target for severity

def to_features(arr):
    z = np.asarray(arr).reshape(arr.shape[0], -1).T
    mag = np.log1p(np.abs(z))
    mag = (mag - mag.mean()) / (mag.std() + 1e-12)
    phase = np.unwrap(np.angle(z), axis=1)
    phase = phase / (np.abs(phase).max() + 1e-12)
    return torch.from_numpy(np.concatenate([mag, phase], axis=0).astype(np.float32))

frf_collection.torch_dataset()
ds = frf_collection.dataset
ds.transform = to_features
n_channels = ds[0][0].shape[0]
print('feature shape:', ds[0][0].shape)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device =', device)


In [ ]:
class FRFClassifier(nn.Module):
    def __init__(self, n_channels, n_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(n_channels, 32, kernel_size=7, padding=3), nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(32, 64, kernel_size=5, padding=2), nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(64, 64, kernel_size=3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(64, 64), nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, n_classes),
        )
    def forward(self, x): return self.net(x)


class FRFRegressor(nn.Module):
    '''Same backbone, single linear output.'''
    def __init__(self, n_channels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(n_channels, 32, kernel_size=7, padding=3), nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(32, 64, kernel_size=5, padding=2), nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(64, 64, kernel_size=3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(64, 64), nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1),
        )
    def forward(self, x): return self.net(x).squeeze(-1)


class HeadDataset(torch.utils.data.Dataset):
    '''Wraps the underlying HDF5Dataset and rewrites the label column.'''
    def __init__(self, base, indices, label_array, dtype=torch.long):
        self.base = base; self.idx = list(indices)
        self.lbl = label_array; self.dtype = dtype
    def __len__(self): return len(self.idx)
    def __getitem__(self, i):
        gi = self.idx[i]
        x, _ = self.base[gi]
        return x, torch.tensor(self.lbl[gi], dtype=self.dtype)


def _train_loop(model, loaders, n_epochs, loss_fn, lr, classify):
    train_loader, val_loader = loaders
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_epochs)
    history = []
    for ep in range(1, n_epochs + 1):
        model.train(True); tot, cor, ls, ae = 0, 0, 0.0, 0.0
        for x, y in train_loader:
            x = x.to(device); y = y.to(device)
            opt.zero_grad(); pred = model(x); loss = loss_fn(pred, y)
            loss.backward(); opt.step()
            ls += loss.item() * x.size(0); tot += x.size(0)
            if classify: cor += (pred.argmax(1) == y).sum().item()
            else: ae += (pred - y).abs().sum().item()
        if classify: tr_metric = cor / tot
        else:        tr_metric = ae / tot
        model.train(False); tot2, cor2, ls2, ae2 = 0, 0, 0.0, 0.0
        with torch.no_grad():
            for x, y in val_loader:
                x = x.to(device); y = y.to(device)
                pred = model(x); loss = loss_fn(pred, y)
                ls2 += loss.item() * x.size(0); tot2 += x.size(0)
                if classify: cor2 += (pred.argmax(1) == y).sum().item()
                else: ae2 += (pred - y).abs().sum().item()
        sched.step()
        if classify: va_metric = cor2 / tot2
        else:        va_metric = ae2 / tot2
        history.append((ls / tot, tr_metric, ls2 / tot2, va_metric))
        if ep == 1 or ep % 5 == 0 or ep == n_epochs:
            tag = 'acc' if classify else 'MAE'
            print(f"  ep {ep:>3d}  tr {tag} {tr_metric:.4f}   val {tag} {va_metric:.4f}")
    return history


def train_classifier(label_array, n_classes, mask, n_epochs=20, lr=2e-3, batch=64):
    tr = [i for i in train_idx if mask[i]]
    va = [i for i in val_idx   if mask[i]]
    te = [i for i in test_idx  if mask[i]]
    train_loader = DataLoader(HeadDataset(ds, tr, label_array, torch.long), batch_size=batch, shuffle=True)
    val_loader   = DataLoader(HeadDataset(ds, va, label_array, torch.long), batch_size=128)
    test_loader  = DataLoader(HeadDataset(ds, te, label_array, torch.long), batch_size=128)
    model = FRFClassifier(n_channels, n_classes).to(device)
    history = _train_loop(model, (train_loader, val_loader), n_epochs,
                            nn.CrossEntropyLoss(), lr, classify=True)
    ys, yps = [], []
    model.eval()
    with torch.no_grad():
        for x, y in test_loader:
            ys.append(y.numpy())
            yps.append(model(x.to(device)).argmax(1).cpu().numpy())
    return model, np.array(history), np.concatenate(ys), np.concatenate(yps)


def train_regressor(target_array, mask, n_epochs=30, lr=2e-3, batch=64):
    tr = [i for i in train_idx if mask[i]]
    va = [i for i in val_idx   if mask[i]]
    te = [i for i in test_idx  if mask[i]]
    train_loader = DataLoader(HeadDataset(ds, tr, target_array, torch.float32), batch_size=batch, shuffle=True)
    val_loader   = DataLoader(HeadDataset(ds, va, target_array, torch.float32), batch_size=128)
    test_loader  = DataLoader(HeadDataset(ds, te, target_array, torch.float32), batch_size=128)
    model = FRFRegressor(n_channels).to(device)
    history = _train_loop(model, (train_loader, val_loader), n_epochs,
                            nn.MSELoss(), lr, classify=False)
    ys, yps = [], []
    model.eval()
    with torch.no_grad():
        for x, y in test_loader:
            ys.append(y.numpy())
            yps.append(model(x.to(device)).cpu().numpy())
    return model, np.array(history), np.concatenate(ys), np.concatenate(yps)


In [ ]:
print("HEAD 1 — DETECTION (pristine vs cracked)")
det_model, det_hist, det_y, det_yp = train_classifier(
    sample_is_dmg.astype(int), 2, np.ones(n, dtype=bool), n_epochs=15)


In [ ]:
print("HEAD 2 — LOCALISATION (regression on x_c / L)")
loc_mask = sample_is_dmg == 1
loc_target = sample_pos.astype(np.float32)
loc_model, loc_hist, loc_y, loc_yp = train_regressor(loc_target, loc_mask, n_epochs=30)


In [ ]:
print("HEAD 3 — SEVERITY (regression on d / h)")
sev_mask = sample_is_dmg == 1
sev_target = sample_depth.astype(np.float32)
sev_model, sev_hist, sev_y, sev_yp = train_regressor(sev_target, sev_mask, n_epochs=30)


## 7. Test-set evaluation

Detection gets a confusion matrix; both regressors get a predicted-vs-true
scatter, with predictions reported in physical units (mm) — distance from
the fixed end for localisation, crack depth for severity.


In [ ]:
from sklearn.metrics import confusion_matrix


def plot_cm(ax, ys, yps, classes, title):
    cm = confusion_matrix(ys, yps, labels=list(range(len(classes))))
    acc = (ys == yps).mean()
    im = ax.imshow(cm, cmap='Blues')
    ax.set_xticks(range(len(classes))); ax.set_yticks(range(len(classes)))
    ax.set_xticklabels(classes, fontsize=10)
    ax.set_yticklabels(classes, fontsize=10)
    ax.set_xlabel('predicted'); ax.set_ylabel('true')
    ax.set_title(f'{title} — acc {acc:.1%}')
    mx = cm.max() if cm.max() else 1
    for i in range(len(classes)):
        for j in range(len(classes)):
            if cm[i, j]:
                ax.text(j, i, cm[i, j], ha='center', va='center',
                         color='white' if cm[i, j] > mx / 2 else 'black', fontsize=11)


def plot_scatter(ax, true, pred, levels, unit_to_mm, ylabel, title):
    '''Scatter predicted vs true at every discrete training level (in mm).'''
    ax.axline((0, 0), (1, 1), color='black', linewidth=0.8, linestyle='--', label='ideal')
    for L in levels:
        sel = np.isclose(true, L, atol=1e-3)
        if not sel.any(): continue
        x = np.full(sel.sum(), L) * unit_to_mm
        y = pred[sel] * unit_to_mm
        ax.scatter(x, y, color='steelblue', alpha=0.45, s=14)
    mae = float(np.abs(pred - true).mean()) * unit_to_mm
    rmse = float(np.sqrt(np.mean((pred - true) ** 2))) * unit_to_mm
    lo = min(true.min(), pred.min()) * unit_to_mm * 0.9
    hi = max(true.max(), pred.max()) * unit_to_mm * 1.1
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
    ax.set_xlabel('true ' + ylabel); ax.set_ylabel('predicted ' + ylabel)
    ax.set_title(f'{title} — MAE {mae:.2f} mm, RMSE {rmse:.2f} mm')
    ax.legend(loc='lower right', fontsize=8)


fig, axes = plt.subplots(1, 3, figsize=(17, 5.2))
plot_cm(axes[0], det_y, det_yp, ['pristine', 'cracked'], 'detection')
plot_scatter(axes[1], loc_y, loc_yp, P.DAMAGE_POSITIONS,
              unit_to_mm=P.BEAM_L * 1000.0,
              ylabel='distance from fixed end [mm]',
              title='localisation')
plot_scatter(axes[2], sev_y, sev_yp, P.DAMAGE_DEPTHS,
              unit_to_mm=P.BEAM_H * 1000.0,
              ylabel='crack depth [mm]',
              title='severity')
plt.tight_layout(); plt.show()

# Per-level summary
print("localisation — per-position test statistics:")
for p in P.DAMAGE_POSITIONS:
    sel = np.isclose(loc_y, p, atol=1e-3)
    if sel.any():
        mu = loc_yp[sel].mean(); sd = loc_yp[sel].std()
        true_mm = p * P.BEAM_L * 1000
        pred_mm = mu * P.BEAM_L * 1000
        sd_mm = sd * P.BEAM_L * 1000
        print(f"  x_c/L = {p:.2f}  true {true_mm:6.1f} mm  pred {pred_mm:6.1f} +/- {sd_mm:4.1f} mm")
print()
print("severity — per-depth test statistics:")
for d in P.DAMAGE_DEPTHS:
    sel = np.isclose(sev_y, d, atol=1e-3)
    if sel.any():
        mu = sev_yp[sel].mean(); sd = sev_yp[sel].std()
        true_mm = d * P.BEAM_H * 1000
        pred_mm = mu * P.BEAM_H * 1000
        sd_mm = sd * P.BEAM_H * 1000
        print(f"  d/h = {d:.2f}  true {true_mm:5.2f} mm  pred {pred_mm:5.2f} +/- {sd_mm:4.2f} mm")


## 8. Swapping in FEA — same notebook, any mesh

`pymodal.scenarios` is mesh-agnostic: only the `BeamGeometry` factory and
the modal provider know what a "beam" is. To repoint the same workflow at
an FEA cantilever (Salome geometry → Code_Aster modal), implement::

```python
def fea_modal_provider(geom, freqs, ins, outs):
    # 1. Re-mesh per realisation (geom.crack_position_frac, geom.crack_depth_frac
    #    drive the slot location and depth in the geometry script).
    # 2. Run Code_Aster modal + harmonic.
    # 3. Stack the response into (n_freq, n_outputs, n_inputs).
    ...

DS.build_dataset(scenarios, ..., modal_provider=fea_modal_provider)
```

For a different structure altogether, replace `BeamGeometry` and the
scenario `apply` callbacks; everything else - the dataset assembler, the
classifier head, the regression heads, the evaluation grid - stays as is.

## Cleanup


In [ ]:
frf_collection.open()
frf_collection.close(keep=True)
print(f"dataset preserved at {dataset_path}")
